# Yoga Pose Classification Using Convolutional Neural Networks and Transfer Learning

## Overview

This notebook develops an image-based yoga pose classification framework using deep learning techniques. Two models are investigated:

* Custom Convolutional Neural Network (CNN)
* MobileNetV2 Transfer Learning

The framework classifies five yoga poses:

1. Downward Dog
2. Goddess
3. Plank
4. Tree
5. Warrior II

The workflow includes:

* Dataset cleaning
* Exploratory analysis
* Data augmentation
* Model development
* Transfer learning
* Performance evaluation
* Error analysis
* Comparative assessment

## Objective

To evaluate the effectiveness of transfer learning compared with a conventional CNN architecture for yoga pose recognition.

## Cell 1: Imports and Configuration

In [ ]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

RAW_BASE = "/kaggle/input/datasets/niharika41298/yoga-poses-dataset/DATASET"
PROJECT_DIR = "/kaggle/working/Yoga-Pose-Recognition"
FIGURE_DIR = os.path.join(PROJECT_DIR, "Figures")
MODEL_DIR = os.path.join(PROJECT_DIR, "Models")
RESULT_DIR = os.path.join(PROJECT_DIR, "Results")
CLEAN_BASE = os.path.join(PROJECT_DIR, "Clean_Dataset")

for folder in [PROJECT_DIR, FIGURE_DIR, MODEL_DIR, RESULT_DIR, CLEAN_BASE]:
    os.makedirs(folder, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
NUM_CLASSES = 5

class_names = ["downdog", "goddess", "plank", "tree", "warrior2"]
display_names = {
    "downdog": "Downward Dog",
    "goddess": "Goddess",
    "plank": "Plank",
    "tree": "Tree",
    "warrior2": "Warrior II"
}

print("Setup completed successfully.")
print("TensorFlow version:", tf.__version__)
print("Dataset path:", RAW_BASE)

## Cell 2: Dataset Cleaning

This cell scans the raw dataset, validates each image, resizes to 160×160, converts to JPEG, and saves cleaned copies. Corrupt or unreadable files are logged to a CSV report.

In [ ]:
# ============================================================
# Cell 3: Clean Dataset
# ============================================================
clean_count = 0
bad_files = []

for split in ["TRAIN", "TEST"]:
    split_path = os.path.join(RAW_BASE, split)
    for cls in class_names:
        in_dir = os.path.join(split_path, cls)
        out_dir = os.path.join(CLEAN_BASE, split, cls)
        os.makedirs(out_dir, exist_ok=True)
        for file in os.listdir(in_dir):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                in_path = os.path.join(in_dir, file)
                out_path = os.path.join(out_dir, os.path.splitext(file)[0] + ".jpg")
                try:
                    img = Image.open(in_path).convert("RGB")
                    img = img.resize(IMG_SIZE)
                    img.save(out_path, "JPEG", quality=95)
                    clean_count += 1
                except Exception as error:
                    bad_files.append({
                        "split": split,
                        "class": cls,
                        "file": file,
                        "error": str(error)
                    })

bad_files_df = pd.DataFrame(bad_files)
bad_files_df.to_csv(
    os.path.join(RESULT_DIR, "bad_files_report.csv"),
    index=False
)

print("Clean images saved:", clean_count)
print("Bad files skipped:", len(bad_files))

## Cell 3: Dataset Statistics

Counts images per class and split, displays a summary table, and saves it as an Excel file.

In [ ]:
# ============================================================
# Cell 4: Dataset Statistics
# ============================================================
records = []
for split in ["TRAIN", "TEST"]:
    for cls in class_names:
        folder = os.path.join(CLEAN_BASE, split, cls)
        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(".jpg")
        ])
        records.append({
            "Split": split,
            "Class": display_names[cls],
            "Number of Images": count
        })

dataset_stats = pd.DataFrame(records)
dataset_stats.to_excel(
    os.path.join(RESULT_DIR, "dataset_statistics.xlsx"),
    index=False
)
display(dataset_stats)
print("Dataset statistics saved.")

## Cell 4: Dataset Characterization Figure

Bar chart showing the number of training images per yoga pose class.

In [ ]:
# ============================================================
# Cell 5: Dataset Characterization Figure
# ============================================================
train_stats = dataset_stats[dataset_stats["Split"] == "TRAIN"]

plt.figure(figsize=(8,5))
bars = plt.bar(
    train_stats["Class"],
    train_stats["Number of Images"]
)
plt.title("Dataset Characterization")
plt.xlabel("Yoga Pose")
plt.ylabel("Number of Training Images")
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 2,
        str(int(height)),
        ha="center"
    )
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "dataset_characterization_dashboard.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Dataset characterization figure saved.")

## Cell 5: Representative Sample Images

Displays one randomly selected training image per class to give a visual overview of the dataset.

In [ ]:
# ============================================================
# Cell 6: Representative Sample Images
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, cls in enumerate(class_names):
    folder = os.path.join(CLEAN_BASE, "TRAIN", cls)
    files = [f for f in os.listdir(folder) if f.lower().endswith(".jpg")]
    sample_file = random.choice(files)
    image_path = os.path.join(folder, sample_file)
    img = Image.open(image_path).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(display_names[cls])
    axes[i].axis("off")

plt.suptitle("Representative Sample Images from the Yoga Pose Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "representative_sample_images.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Representative sample images figure saved.")

## Cell 6: TensorFlow Data Pipeline

Builds `tf.data` pipelines for training (80%), validation (20%), and test sets with prefetching for performance.

In [ ]:
# ============================================================
# Cell 7: TensorFlow Data Pipeline
# ============================================================
train_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_BASE, "TRAIN"),
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
val_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_BASE, "TRAIN"),
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
test_data = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CLEAN_BASE, "TEST"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(AUTOTUNE)
val_data = val_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)

print("Training batches:", len(train_data))
print("Validation batches:", len(val_data))
print("Test batches:", len(test_data))
print("Data pipeline completed successfully.")

## Cell 7: Data Augmentation Examples

Applies random horizontal flip, rotation (±10°), zoom (±10%), and translation (±10%) to a sample image. Displays the original alongside four augmented variants.

In [ ]:
# ============================================================
# Cell 8: Data Augmentation Examples
# ============================================================
augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.10, 0.10)
])

sample_folder = os.path.join(CLEAN_BASE, "TRAIN", "tree")
sample_file = random.choice([
    f for f in os.listdir(sample_folder)
    if f.lower().endswith(".jpg")
])
sample_path = os.path.join(sample_folder, sample_file)
img = Image.open(sample_path).convert("RGB")
img_array = np.array(img)

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(img_array)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1, 5):
    augmented_img = augmentation(
        tf.expand_dims(img_array, 0),
        training=True
    )
    axes[i].imshow(augmented_img[0].numpy().astype("uint8"))
    axes[i].set_title("Augmented")
    axes[i].axis("off")

plt.suptitle("Data Augmentation Examples", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "data_augmentation_examples.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Data augmentation figure saved.")

## Cell 8: Custom CNN — Architecture and Training

A 3-block convolutional network (32→64→128 filters) with a dense head. Trained for up to 8 epochs with early stopping (patience=3) and best-model checkpointing monitored on validation accuracy.

In [ ]:
# ============================================================
# Cell 9: Custom CNN Model and Training
# ============================================================
custom_cnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

custom_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
custom_cnn.summary()

callbacks_cnn = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, "custom_cnn_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger(
        os.path.join(RESULT_DIR, "custom_cnn_training_log.csv"),
        append=False
    )
]

history_cnn = custom_cnn.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=callbacks_cnn
)

custom_cnn.save(
    os.path.join(MODEL_DIR, "custom_cnn_final.keras")
)
print("Custom CNN training completed and saved.")

## Cell 9: MobileNetV2 Transfer Learning

Uses MobileNetV2 pre-trained on ImageNet as a frozen feature extractor. A Global Average Pooling layer, dropout (0.4), and two dense layers are added on top. Trained for up to 15 epochs with early stopping (patience=5).

In [ ]:
# ============================================================
# Cell 10: MobileNetV2 Transfer Learning
# ============================================================
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

mobilenet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

mobilenet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
mobilenet_model.summary()

callbacks_mobilenet = [
    ModelCheckpoint(
        os.path.join(MODEL_DIR, "mobilenetv2_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger(
        os.path.join(RESULT_DIR, "mobilenetv2_training_log.csv"),
        append=False
    )
]

history_mobilenet = mobilenet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=callbacks_mobilenet
)

mobilenet_model.save(
    os.path.join(MODEL_DIR, "mobilenetv2_final.keras")
)
print("MobileNetV2 training completed and saved.")

## Cell 10: Training Curves

Plots accuracy and loss curves for both models across epochs. All figures are saved as high-resolution PDFs.

In [ ]:
# ============================================================
# Cell 11: Training Curves
# ============================================================
custom_history_df = pd.DataFrame(history_cnn.history)
mobilenet_history_df = pd.DataFrame(history_mobilenet.history)

custom_history_df.to_excel(
    os.path.join(RESULT_DIR, "custom_cnn_history.xlsx"),
    index=False
)
mobilenet_history_df.to_excel(
    os.path.join(RESULT_DIR, "mobilenetv2_history.xlsx"),
    index=False
)

plt.figure(figsize=(7,5))
plt.plot(custom_history_df["accuracy"], label="Training Accuracy")
plt.plot(custom_history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Custom CNN Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "custom_cnn_accuracy_curve.pdf"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,5))
plt.plot(custom_history_df["loss"], label="Training Loss")
plt.plot(custom_history_df["val_loss"], label="Validation Loss")
plt.title("Custom CNN Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "custom_cnn_loss_curve.pdf"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,5))
plt.plot(mobilenet_history_df["accuracy"], label="Training Accuracy")
plt.plot(mobilenet_history_df["val_accuracy"], label="Validation Accuracy")
plt.title("MobileNetV2 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "mobilenetv2_accuracy_curve.pdf"), dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,5))
plt.plot(mobilenet_history_df["loss"], label="Training Loss")
plt.plot(mobilenet_history_df["val_loss"], label="Validation Loss")
plt.title("MobileNetV2 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "mobilenetv2_loss_curve.pdf"), dpi=300, bbox_inches="tight")
plt.show()

print("Training curves saved and displayed.")

## Cell 11: Final Evaluation — Classification Report

Evaluates MobileNetV2 on the test set and generates a full classification report (precision, recall, F1-score per class). Results are displayed and saved to Excel.

In [ ]:
# ============================================================
# Cell 12: Final Evaluation
# ============================================================
y_true = np.concatenate([y for x, y in test_data], axis=0)
y_pred_prob = mobilenet_model.predict(test_data)
y_pred = np.argmax(y_pred_prob, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=[display_names[c] for c in class_names],
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()
display(report_df)
report_df.to_excel(
    os.path.join(RESULT_DIR, "mobilenetv2_classification_report.xlsx")
)
print("Classification report saved.")

## Cell 12: Confusion Matrix

Visualises the confusion matrix for MobileNetV2 predictions on the test set. Each cell shows the count of true vs predicted labels.

In [ ]:
# ============================================================
# Cell 13: Confusion Matrix
# ============================================================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap="Blues")
plt.title("MobileNetV2 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(
    range(len(class_names)),
    [display_names[c] for c in class_names],
    rotation=45
)
plt.yticks(
    range(len(class_names)),
    [display_names[c] for c in class_names]
)
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )
plt.tight_layout()
plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "mobilenetv2_confusion_matrix.pdf"
    ),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Confusion matrix saved.")

## Cell 13: Model Comparison

Evaluates both models on the test set and presents a side-by-side comparison of test accuracy and loss. Results are saved to Excel and visualised as a bar chart.

In [ ]:
# ============================================================
# Cell 14: Model Comparison
# ============================================================
custom_loss, custom_acc = custom_cnn.evaluate(
    test_data,
    verbose=0
)
mobile_loss, mobile_acc = mobilenet_model.evaluate(
    test_data,
    verbose=0
)

comparison_df = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2"],
    "Test Accuracy": [
        custom_acc * 100,
        mobile_acc * 100
    ],
    "Test Loss": [
        custom_loss,
        mobile_loss
    ]
})
display(comparison_df)
comparison_df.to_excel(
    os.path.join(
        RESULT_DIR,
        "model_comparison.xlsx"
    ),
    index=False
)

plt.figure(figsize=(7,5))
bars = plt.bar(
    comparison_df["Model"],
    comparison_df["Test Accuracy"]
)
plt.ylabel("Accuracy (%)")
plt.title("Model Performance Comparison")
for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.1f}%",
        ha="center"
    )
plt.tight_layout()
plt.savefig(
    os.path.join(
        FIGURE_DIR,
        "model_performance_comparison.pdf"
    ),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print(comparison_df)

## Cell 14: Class-wise F1 Scores

Extracts per-class F1 scores from the classification report and visualises them as a bar chart, providing insight into which poses are most and least accurately classified.

In [ ]:
# ============================================================
# Cell 16: Class-wise F1 Scores
# ============================================================
f1_df = report_df.loc[
    [display_names[c] for c in class_names],
    ["f1-score"]
].reset_index()
f1_df.columns = ["Class", "F1 Score"]
display(f1_df)
f1_df.to_excel(
    os.path.join(RESULT_DIR, "classwise_f1_scores.xlsx"),
    index=False
)

plt.figure(figsize=(8,5))
bars = plt.bar(
    f1_df["Class"],
    f1_df["F1 Score"]
)
plt.title("Class-wise F1 Scores for MobileNetV2")
plt.xlabel("Yoga Pose")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "classwise_f1_scores.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Class-wise F1 score figure saved.")

## Cell 15: Export Project

Generates a `README.md` summarising the project and packages all outputs (figures, models, results) into a ZIP archive for download.

In [ ]:
# ============================================================
# Cell 17: Create README and Export Project
# ============================================================
readme_text = f"""
# Yoga Pose Classification Using Convolutional Neural Networks and Transfer Learning

## Project Overview
This project classifies five yoga poses using deep learning:
- Custom CNN
- MobileNetV2 Transfer Learning

## Yoga Pose Classes
- Downward Dog
- Goddess
- Plank
- Tree
- Warrior II

## Final Model Performance
| Model | Test Accuracy (%) | Test Loss |
|---|---:|---:|
| Custom CNN | {custom_acc * 100:.2f} | {custom_loss:.4f} |
| MobileNetV2 | {mobile_acc * 100:.2f} | {mobile_loss:.4f} |

## Repository Contents
- `Figures/`: PDF result figures
- `Models/`: Trained Keras models
- `Results/`: Excel and CSV result files
- `Notebook/`: Kaggle notebook with code and visible outputs

## Key Outputs
- Dataset characterization
- Data augmentation examples
- CNN and MobileNetV2 training curves
- Confusion matrix
- Classification report
- Class-wise F1 score analysis
- Detailed mixed prediction analysis
"""

with open(os.path.join(PROJECT_DIR, "README.md"), "w") as f:
    f.write(readme_text)

zip_path = "/kaggle/working/Yoga-Pose-Recognition.zip"
shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    PROJECT_DIR
)
print("README created.")
print("Project ZIP created:")
print(zip_path)